# Colab Worker A: Track A - VAE Transfer Sweeps & Wiener Fit
This notebook runs on **Colab Account 1** (T4/V100/A100 GPU).
- Measures empirical transfer function $H(\text{band})$ across CompressAI & SD VAE
- Fits natural image power spectrum $P(f) = A \cdot f^{-\alpha}$
- Fits Wiener model parameter $N_{\text{eff}}$ and predicts theoretical cutoff $f_c$
- Exports `results/fitted_g.json` (Merge 1 input for Worker C)

In [ ]:
# 1. Check GPU and mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/results

In [ ]:
# 2. Clone repository & install dependencies
!git clone https://github.com/<YOUR_GITHUB_REPO>.git /content/repo
%cd /content/repo
!pip install -q diffusers transformers accelerate compressai kornia lpips pandas tqdm

In [ ]:
# 3. Unpack processed image splits (fit 300 & test 500)
!unzip -q /content/repo/data/processed_splits.zip -d /content/repo/data/processed/

In [ ]:
# 4. Run Track A Transfer Function Sweeps
!python -m track_a.measure \
    --manifest data/manifests/fit_manifest.json \
    --images_dir data/processed/fit \
    --out results/track_a_transfer.csv \
    --attacks compressai sd_vae \
    --limit 300

!cp results/track_a_transfer.csv $SHARED_DIR/results/

In [ ]:
# 5. Fit Image Power Spectrum & Wiener Model
!python -m track_a.fit \
    --manifest data/manifests/fit_manifest.json \
    --images_dir data/processed/fit \
    --transfer_csv results/track_a_transfer.csv \
    --out_json results/fitted_g.json \
    --psnr 40.0

# Export fitted_g.json to Google Drive for Worker C
!cp results/fitted_g.json $SHARED_DIR/results/
print('✓ Track A completed successfully! fitted_g.json ready for Merge 1.')